# Calibrate using SciPy Gradient Descent

In [34]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

import fates_calibration_library.emulator_functions as em
import fates_calibration_library.utils as utils
from fates_calibration_library.TFClass import TFEmulator

import importlib

## Functions

In [28]:
def load_parameter_metadata(ensemble_config):
    
    # load Latin Hypercube key to get number of parameters
    lhc_key = pd.read_csv(ensemble_config['lhc_key_file'], index_col=[0])
    lhc_key = lhc_key.drop(columns=['ensemble'])
    param_names = lhc_key.columns
    num_params = len(param_names)
    
    # get normalized default parameter values
    default_norm = pd.read_csv(ensemble_config['default_norm'], index_col=[0])
    
    return param_names, num_params, default_norm

def get_pft_info(pft, default_param_file, pft_id_config):
    
    pft_ids = utils.get_config_file(pft_id_config)
    default_param = xr.open_dataset(default_param_file)
    all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]
    pft_name = all_pfts[pft-1]
    pft_id = pft_ids[pft_name]
    
    return pft_name, pft_id

def load_emulator_and_obs_data(ensemble_config, pft_name, pft_id, emulator_dir, calibration_vars,
                              obs_config_file):
    
    # load observations
    obs = pd.read_csv(ensemble_config['obs_df'], index_col=[0])
    obs_pft = obs[obs.pft == pft_name]
    obs_pft = obs_pft[obs_pft.land_frac > 0.99]
    if pft_id != 'AC3G':
        obs_pft = obs_pft[obs_pft.pct_lake < 30]
    
    # load parameter sensitivity
    sens_df = pd.read_csv(ensemble_config['sens_df'], index_col=[0])
    sens_pft = sens_df[sens_df.pft == pft_id]
    
    obs_config = utils.get_config_file(obs_config_file)
    emulators, targets, sds = em.prep_calibration_data(obs_pft, calibration_vars, obs_config, 
                                                    emulator_dir, pft_name, pft_id)
    
    return emulators, targets, sds, sens_pft

def build_optimization_config(ensemble_config):
    
    # build config
    config = {
        'maxiter': ensemble_config['maxiter'],
        'epsilon': 0.5,
        'lambda_penalty': None,
        'barrier_strength': 0,
        'loss_fn': em.squared_z_loss,
        'default_penalty_fn': em.default_penalty_l1,
        'barrier_penalty_fn': em.barrier_penalty,
        'tol': 1e-3
    }
    
    return config

## Set Up
Load files, set up ensemble information

In [41]:
config_file = '/glade/work/afoster/FATES_calibration/emulator_configs/dom_pft.yaml'
ensemble_config = utils.get_config_file(config_file)

calib_var_file = '/glade/work/afoster/FATES_calibration/emulator_configs/calibration_vars.yaml'
calib_vars_config = utils.get_config_file(calib_var_file)

param_update_file = '/glade/work/afoster/FATES_calibration/emulator_configs/param_min_max.yaml'
param_update_config = utils.get_config_file(param_update_file)

pft_id_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_pft_ids.yaml'

obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'

emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'

sobol = 0.01

In [42]:
# parameter information
param_names, num_params, default_norm = load_parameter_metadata(ensemble_config)

## Chose PFT

In [43]:
pft = 14

In [44]:
# pft information
pft_name, pft_id = get_pft_info(pft, ensemble_config['default_param'], pft_id_config)

## Load required objects

In [45]:
# emulators and targets
emulators, targets, sds, sens_pft = load_emulator_and_obs_data(
    ensemble_config, pft_name, pft_id, emulator_dir, calib_vars_config[pft_id],
    obs_config_file
)

In [46]:
# default parameters
params_default = em.get_default_pft_values(default_norm, pft)

In [47]:
# indices for to fix and optimize
fixed_indices, optimize_indices, num_optimize = em.get_params_to_optimize(sens_pft,
                                                                          param_names, 
                                                                          num_params, sobol_threshold=sobol)

In [48]:
# optimization config
config = build_optimization_config(ensemble_config)

## Calibration

In [55]:
# all_results = em.run_batch_optimization(emulators, targets, sds, fixed_indices, params_default, num_optimize, 
#                        param_names, optimize_indices, config, param_update_config, 
#                        num_batch=200)

In [56]:
optimize_pars = param_names[optimize_indices]

In [57]:
optimize_pars

Index(['fates_allom_fnrt_prof_b', 'fates_leaf_slatop',
       'fates_leaf_stomatal_intercept', 'fates_leaf_stomatal_slope_medlyn',
       'fates_leaf_vcmax25top', 'fates_leafn_vert_scaler_coeff2',
       'fates_maintresp_leaf_ryan1991_baserate', 'smpsc_delta',
       'fates_rad_leaf_clumping_index', 'fates_rad_leaf_xl',
       'fates_stoich_nitr_1', 'fates_turb_leaf_diameter', 'fates_turb_z0mr'],
      dtype='object')

In [59]:
result = em.run_optimization(emulators, targets, sds, fixed_indices, optimize_pars,
                          params_default, num_optimize, config, param_update_config)

In [60]:
result

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 19.124238844553016
        x: [ 5.543e-01  6.278e-01 ...  1.350e-02  7.388e-01]
      nit: 1
      jac: [ 2.150e+02  4.978e+02 ...  3.916e+02  2.321e+02]
     nfev: 224
     njev: 16
 hess_inv: <13x13 LbfgsInvHessProduct with dtype=float64>